In [9]:
import os, sys, math, ssl, io, pytz, numpy as np, pandas as pd, requests
from datetime import datetime, timedelta, date
from timezonefinder import TimezoneFinder
from meteostat import Stations, Hourly
from isd import Batch
from scp import SCPClient
import paramiko
import calendar
from pandas.errors import EmptyDataError




def convert_utc_to_local(df, local_tz):
    """
    Convert the datetime index of the DataFrame from UTC to a local timezone.

    Args:
    df : pandas.DataFrame
        DataFrame with a datetime index in UTC.
    local_tz : str
        A timezone string (e.g., 'America/Chicago').

    Returns:
    pandas.DataFrame
        DataFrame with datetime index converted to the specified local timezone.
    """
    if not pd.api.types.is_datetime64_any_dtype(df.index):
        df = df.reset_index(level='station', drop=True)
        df.index = pd.to_datetime(df.index)

    if df.index.tz is None:
        df.index = df.index.tz_localize('UTC')
    
    df.index = df.index.tz_convert(local_tz)
    return df

def filter_dataframe_by_date(df, start_date, end_date, timezone=None):
    """
    Filter the DataFrame to include rows between the specified start and end dates,
    handling timezone differences appropriately.
    """
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    if timezone:
        start_date = start_date.tz_localize(timezone)
        end_date = end_date.tz_localize(timezone)
    else:
        df.index = df.index.tz_localize(None)

    return df.loc[(df.index >= start_date) & (df.index <= end_date)]

def get_parameters_MERRA2(lat, lon, year):
    api_endpoint = f"https://power.larc.nasa.gov/api/temporal/hourly/point?community=SB&parameters=&longitude={lon}&latitude={lat}&start={year}0101&end={year}1231&format=EPW"
    response = requests.get(api_endpoint)
    csv_data = io.StringIO(response.text)
    df = pd.read_csv(csv_data, skiprows=8, header=None)
    header = '\n'.join(response.text.splitlines()[:8])

    # Check if the dataframe has more than 8761 rows and truncate if necessary
    # Sometimes MERRA2 erroneously provides extra rows

    if calendar.isleap(int(year)):
        df = df.iloc[:8784]
    else:
        df = df.iloc[:8760]        

    return df, header

def merge_data(df, data):
    """
    Merge data into the DataFrame, interpolating small gaps and filling large gaps with custom values.

    Args:
        df (pd.DataFrame): The target DataFrame.
        data (dict): The source data dictionary.
    Returns:
        pd.DataFrame: The updated DataFrame.
    """
    # Custom fill values for each column
    fill_values = {
        6: 99.9,   # Dry bulb temperature
        7: 99.9,   # Dew point temperature
        8: 999,   # Relative humidity
        33: 999,   # Precipitation
        30: 999,   # Snow
        21: 999,  # Wind speed
        20: 999,  # Wind direction
        9: 999999    # Pressure
    }
    # Define the columns to process and their corresponding data keys
    columns_to_process = {
        6: 'temp',    # Dry bulb temperature
        7: 'dwpt',    # Dew point temperature
        8: 'rhum',    # Relative humidity
        33: 'prcp',   # Precipitation
        30: 'snow',   # Snow
        21: 'wspd',   # Wind speed
        20: 'wdir',   # Wind direction
        9: 'pres'     # Pressure
    }

    for col, key in columns_to_process.items():
        if not data[key].isna().all():
            # Convert the data to a pandas Series for interpolation
            series = pd.Series(list(data[key][1:]))
            
            # Identify gaps (missing values)
            is_missing = series.isna()
            
            # Find consecutive gaps
            gap_groups = is_missing.ne(is_missing.shift()).cumsum()
            gap_sizes = is_missing.groupby(gap_groups).transform('size')
            
            # Interpolate small gaps (<= 3 hours)
            series_interpolated = series.interpolate(method='linear', limit=3, limit_direction='both')
            
            # Fill large gaps (> 3 hours) with the custom fill value
            series_interpolated[gap_sizes > 3] = fill_values.get(col, None)  # Use None as default if no fill value is provided
            
            # Update the DataFrame column with the interpolated and filled data
            df[col] = series_interpolated.tolist()

    return df

def check_missing_hours(year, df):
    """
    Checks for missing hours in the DataFrame's datetime index for a specified year.
    """
    full_index = pd.date_range(start=f"{year}-01-01", end=f"{year+1}-01-01", freq="H")
    missing_hours = full_index.difference(df.index)
    missing_hours_num = len(missing_hours)

    if missing_hours_num > 0:
        diffs = missing_hours.to_series().diff().dt.total_seconds().div(3600)
        largest_consecutive_group = (diffs != 1).cumsum().value_counts().max()
    else:
        largest_consecutive_group = 0

    return missing_hours_num, largest_consecutive_group

def get_noaa_merra2_data(lat, lon, year, file_type, save_folder):
    """
    Retrieves NOAA and MERRA2 data for a specific location and year.
    
    Returns:
        df_merged (DataFrame or str): Merged weather data or empty string if unavailable.
        retrieve_status (bool): Status of data retrieval.
        info_dict (dict or str): Metadata dictionary or empty string if unavailable.
        hdd (float or str): Heating Degree Days or empty string if unavailable.
        cdd (float or str): Cooling Degree Days or empty string if unavailable.
        wmo (str): WMO station identifier or empty string if unavailable.
        latitude_station (float or str): Station latitude or empty string if unavailable.
        longitude_station (float or str): Station longitude or empty string if unavailable.
        epw_exists (bool): Flag indicating if EPW data already exists.
    """
    # Retrieve NOAA data
    data_noaa, tz, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists, incomplete_timeseries = get_data_noaa(lat, lon, year, save_folder)
    
    # If EPW file exists, return early
    if epw_exists:
        return "", False, "", "", "", wmo, "", "", True

    # If timeseries is incomplete, return early
    if incomplete_timeseries:
        return "", False, "", "", "", "", "", "", False

    try:
        # Convert NOAA data to local timezone and filter by date
        data_noaa_tz_adj = filter_dataframe_by_date(
            convert_utc_to_local(data_noaa, tz), 
            datetime(year, 1, 1), 
            datetime(year+1, 1, 1)
        )
    except AttributeError:
        return "", False, "", "", "", "", "", "", False

    # Create metadata dictionary
    info_dict = {
        "timeshift": get_time_shift(tz),
        "elevation": elevation,
        "wmo": wmo,
        "station_name": station_name,
        "state": state,
        "country": country,
        "lat": latitude_station,
        "lon": longitude_station,
        "weather_file_type": file_type
    }

    # Resample and interpolate missing hourly values
    data_noaa_tz_adj_h = data_noaa_tz_adj.resample("H").mean()
    data_noaa_tz_adj_h_interpolated = data_noaa_tz_adj_h.interpolate(method="linear", limit=3, limit_direction="both")

    # Calculate HDD and CDD
    hdd, cdd = calculate_hdd_cdd(data_noaa_tz_adj_h_interpolated, "temp")

    # Attempt to retrieve MERRA2 data with retry logic
    df_merra2, header_merra2 = None, None
    for _ in range(5):  # Try up to 5 times
        try:
            df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
            break  # Exit loop if successful
        except EmptyDataError:
            continue

    if df_merra2 is None:
        return "", False, "", "", "", wmo, latitude_station, longitude_station, False

    # Merge NOAA and MERRA2 data
    df_merged = merge_data(df_merra2, data_noaa_tz_adj_h_interpolated)

    # Ensure no missing data in merged dataframe
    if df_merged.isnull().any().any():
        raise ValueError("The merged DataFrame (df_merged) contains empty cells. Stopping execution.")

    return df_merged, True, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, False

def run_individual_location(lat, lon, year, file_type, save_folder, save_name):
    """
    Processes a single location, fetching data and handling errors.
    
    Returns:
        retrieve_status (bool): Whether the data was successfully retrieved.
        wmo (str): WMO station identifier.
        hdd (float or str): Heating Degree Days or empty string.
        cdd (float or str): Cooling Degree Days or empty string.
        latitude_station (float or str): Latitude of the station.
        longitude_station (float or str): Longitude of the station.
        retrieve_info_closest_other_locations (bool): Whether to attempt retrieving data from other locations.
    """
    data_meteostat_merra2, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists = get_noaa_merra2_data(lat, lon, year, file_type, save_folder)
    
    # If EPW file exists, do not proceed with further data retrieval
    if epw_exists:
        return False, wmo, "", "", "", "", True

    # If data retrieval was successful, save the EPW file
    if retrieve_status:
        retrieve_info_closest_other_locations = False

        # Determine the output file path
        output_filename = f"{save_name.replace(' ', '_').replace('.', '_')}_{year}.epw" if save_name else f"{wmo}_{year}.epw"
        output_path = os.path.join(save_folder, output_filename)

        # Save data as CSV (EPW format)
        data_meteostat_merra2.to_csv(output_path, header=False, index=False)

        # Modify file with EPW header
        with open(output_path, "r") as original_file:
            data_content = original_file.read()

        header_lines = create_header(data_meteostat_merra2, year, info_dict)

        with open(output_path, "w") as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)
    else:
        retrieve_info_closest_other_locations = False
        retrieve_status = False
        print("No data available for this location/year.")

    return retrieve_status, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations

def get_time_shift(timezone_name):
    """
    Calculates the time shift for a given timezone from UTC.
    """
    timezone = pytz.timezone(timezone_name)
    now = datetime.now(timezone)
    utc_offset = now.utcoffset()
    return int(utc_offset.total_seconds() // 3600)  # Return hours offset only

def calculate_hdd_cdd(df, temperature_column):
    """
    Calculate Heating Degree Days (HDD) and Cooling Degree Days (CDD) from hourly temperature data in Celsius.
    """
    df[temperature_column + '_F'] = df[temperature_column] * 9 / 5 + 32
    base_temperature = 65

    df['date'] = df.index.to_series().dt.date
    daily_mean_temp = df.groupby('date')[temperature_column + '_F'].mean().reset_index()
    daily_mean_temp.columns = ['date', 'mean_temp']

    daily_mean_temp['HDD'] = (base_temperature - daily_mean_temp['mean_temp']).clip(lower=0)
    daily_mean_temp['CDD'] = (daily_mean_temp['mean_temp'] - base_temperature).clip(lower=0)

    total_hdd = daily_mean_temp['HDD'].sum()
    total_cdd = daily_mean_temp['CDD'].sum()

    return int(total_hdd), int(total_cdd)

def check_epw_exists(save_folder, year, wmo):
    return os.path.exists(f'{save_folder}/{wmo}_{year}.epw')

def calc_combined_ground_temperatures(df):
    """
    Calculate shallow ground temperatures for multiple depths using the Kusuda and Achenbach model
    and format results as a single EPW GROUND TEMPERATURES line.

    Parameters:
        df (pd.DataFrame): DataFrame with hourly temperatures in column 6.

    Returns:
        str: Combined GROUND TEMPERATURES line in EPW file format for all depths.
    """
    depths = [0.5, 2, 4]  # Depths in meters

    # Conversion function
    def convert(value, from_unit, to_unit):
        conversions = {
            ("yr", "hr"): lambda x: x * 365.25 * 24,  # Convert 1 year to hours
            ("C", "R"): lambda x: (x + 273.15) * 9 / 5,  # Celsius to Rankine
            ("R", "C"): lambda x: (x - 491.67) * 5 / 9,  # Rankine to Celsius
            ("C", "F"): lambda x: x * 9 / 5 + 32,  # Celsius to Fahrenheit
            ("F", "C"): lambda x: (x - 32) * 5 / 9,  # Fahrenheit to Celsius
            ("R", "F"): lambda x: x - 459.67,  # Rankine to Fahrenheit
        }
        return conversions.get((from_unit, to_unit), lambda x: ValueError(f"Unsupported conversion {from_unit} to {to_unit}"))(value)

    # Ensure proper datetime index
    df.index = pd.to_datetime(df[[0, 1, 2, 3]].rename(columns={0: 'year', 1: 'month', 2: 'day', 3: 'hour'}))

    # Constants
    amon = np.array([15, 46, 74, 95, 135, 166, 196, 227, 258, 288, 319, 349])  # Approximate mid-month days
    po = 0.6  # Phase offset
    dif = 0.025  # Thermal diffusivity (m²/hr)
    p = convert(1.0, "yr", "hr")  # Convert 1 year to hours

    # Rename column 6 to 'Temperature'
    df.rename(columns={6: "Temperature (°C)"}, inplace=True)

    # Calculate monthly and annual average temperatures in Celsius
    monthly_avg_temp_c = df.resample("M")["Temperature (°C)"].mean()
    annual_avg_temp_c = df["Temperature (°C)"].mean()

    # Convert averages to Rankine
    monthly_avg_temp_r = monthly_avg_temp_c.apply(lambda x: convert(x, "C", "R"))
    annual_avg_temp_r = convert(annual_avg_temp_c, "C", "R")

    # Prepare the GROUND TEMPERATURES line
    combined_ground_temperatures = ["GROUND TEMPERATURES", str(len(depths))]

    for depth in depths:
        # Kusuda and Achenbach parameters
        beta = math.sqrt(math.pi / (p * dif)) * 10
        x, s, c = math.exp(-beta), math.sin(beta), math.cos(beta)
        y = (x**2 - 2 * x * c + 1) / (2 * beta**2)
        gm = math.sqrt(y) * math.exp(-depth * math.sqrt(math.pi / (p * dif)))
        z, phi = (1 - x * (c + s)) / (1 - x * (c - s)), math.atan((1 - x * (c + s)) / (1 - x * (c - s)))
        bo = 0.5 * (monthly_avg_temp_r.max() - monthly_avg_temp_r.min())

        # Compute ground temperatures for each month
        shallow_ground_monthly_temps_r = annual_avg_temp_r - bo * np.cos(2 * np.pi / p * amon * 24 - po - phi) * gm
        shallow_ground_monthly_temps_c = [convert(temp, "R", "C") for temp in shallow_ground_monthly_temps_r]

        # Append depth and temperatures
        combined_ground_temperatures.extend([f"{depth:.1f}", "", "", ""] + [f"{temp:.2f}" for temp in shallow_ground_monthly_temps_c])

    return ",".join(combined_ground_temperatures)

def create_header(df, year, info_dict):
    header_lines = []

    #Calculated parameters 
    first_day_year = pd.to_datetime(date.min.replace(year=year)).day_name()
    leap_status = lambda year: 'Yes' if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 'No'
    dst_start, dst_end = get_dst_start_end(year, info_dict['lat'], info_dict['lon'])
    design_conditions_file = 'resources/design_conditions.csv'
    design_conditions_line = find_closest_design_condition(float(info_dict['lat']), float(info_dict['lon']), design_conditions_file)
    ground_temp_line = calc_combined_ground_temperatures(df)
    #Hardcoded parameters
    number_of_holidays = 0
    number_of_data_periods = 1
    number_of_records_per_hour = 1

    # line_1
    header_lines.append(f"LOCATION,{info_dict['station_name']},{info_dict['state']},{info_dict['country']},{info_dict['weather_file_type']},{info_dict['wmo']},{info_dict['lat']},{info_dict['lon']},{info_dict['timeshift']},{info_dict['elevation']}")
    # line_2
    header_lines.append(design_conditions_line)
    # line_3
    header_lines.append(f"TYPICAL/EXTREME PERIODS,0")
    # line_4
    header_lines.append(ground_temp_line)
    # header_lines.append(f"GROUND TEMPERATURES,0")
    # header_lines.append(f"GROUND TEMPERATURES,3,.5,,,,-16.34,-17.80,-15.22,-11.16,-0.57,7.61,13.13,14.81,11.95,5.60,-2.89,-10.76,2,,,,-10.97,-13.57,-13.04,-10.89,-3.80,2.61,7.74,10.49,9.90,6.30,0.46,-5.74,4,,,,-6.53,-9.19,-9.78,-8.97,-4.96,-0.64,3.32,6.08,6.72,5.16,1.73,-2.4")
    # line_5
    try:
        header_lines.append(f"HOLIDAYS/DAYLIGHT SAVINGS,{leap_status(year)},{dst_start.month}/{dst_start.day},{dst_end.month}/{dst_end.day},{number_of_holidays}")
    except AttributeError:
        #We cannot retrieve DST dates, let's set them to 0
        header_lines.append(f"HOLIDAYS/DAYLIGHT SAVINGS,{leap_status(year)},0,0,{number_of_holidays}")
    # line_6
    header_lines.append(f"COMMENTS 1, ")
    # line_7
    header_lines.append(f"COMMENTS 2, ")
    # line_8
    header_lines.append(f"DATA PERIODS,{number_of_data_periods},{number_of_records_per_hour},Data,{first_day_year},{df.iloc[0, 1]}/{df.iloc[0, 2]},{df.iloc[-1, 1]}/{df.iloc[-1, 2]}")

    return header_lines

def fix_wmo(wmo):
    """
    Attempts to fix or standardize the WMO code format.
    """
    try:
        return str(int(wmo))
    except ValueError:
        icao = wmo
        icao_converted = get_wmo_from_icao_NOAA(icao)
        if isinstance(icao_converted, type(None)):
            return icao
        else:
            return icao_converted


def get_data_noaa(lat, lon, year, save_folder):
    """
    Fetches NOAA data for a given location and year, handling timezones and missing data.
    """
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year - 1, 12, 31)
    end = datetime(year + 1, 1, 2)

    stations = Stations().nearby(lat, lon)

    epw_exists = False
    station_number = 0
    len_data = 0

    incomplete_timeseries = True
    while incomplete_timeseries:
        station_number += 1
        wmo = fix_wmo(str(stations.fetch(station_number).index.values[-1]))
        # First check if EPW already exists
        if check_epw_exists(save_folder, year, wmo):
            epw_exists = True
            incomplete_timeseries = False
            break
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        if (len(data.index) >100) & (station_number>1):
            data = data.loc[data.index.get_level_values('station').unique()[-1]]

        len_data = len(data.index)
        missing_hours_num, largest_consecutive_group = check_missing_hours(year, data)
        if (len_data > 8000) & (largest_consecutive_group <= 3):
            incomplete_timeseries = False

    if epw_exists | incomplete_timeseries:
        data = ''
        timezone = ''
        # distance = ''
        elevation = ''
        station_name = ''
        state = ''
        country = ''
        latitude_station = ''
        longitude_station = ''
                
    else:
        station_info = stations.fetch(station_number)
        timezone = station_info['timezone'].values[-1]
        elevation = station_info['elevation'].values[-1]
        # distance = stations.fetch()['distance'].values[-1]
        wmo = fix_wmo(str(station_info.index.values[-1]))
        station_name = station_info['name'].values[-1]
        state = station_info['region'].values[-1]
        country = station_info['country'].values[-1]
        latitude_station = station_info['latitude'].values[-1]
        longitude_station = station_info['longitude'].values[-1]

    # return data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries
    return data, timezone, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries




def update_if_missing(df, index, col_name, new_value):
    if pd.isna(df.at[index, col_name]) or not df.at[index, col_name]:
        df.at[index, col_name] = new_value

def retrieve_info_other_location(wmo, zipcodes, year):
    retrieve_status = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"EPW_file_name_{year}"].values[0]
    # distance = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"distance_location_station_miles_{year}"].values[0]
    hdd = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"hdd_base65F_{year}"].values[0]
    cdd = zipcodes[zipcodes[f"weather_station_wmo_{year}"]==str(wmo)][f"cdd_base65F_{year}"].values[0]
    # return retrieve_status, distance, hdd, cdd
    return retrieve_status, hdd, cdd

def get_wmo_from_icao_NOAA(icao_code):
    # Path to the local CSV file in the resource folder
    csv_file_path = os.path.join(os.path.join(os.getcwd(), 'resources'), 'isd-history.csv')

    # Read the CSV file
    try:
        with open(csv_file_path, 'r', encoding='utf-8') as file:
            lines = file.readlines()
            headers = lines[0].split(',')
            icao_index = headers.index('"ICAO"')
            wmo_index = headers.index('"USAF"')

            for line in lines[1:]:
                fields = line.split(',')
                if fields[icao_index].strip('"') == icao_code.upper():
                    return fields[wmo_index].strip('"')

    except FileNotFoundError:
        print(f"CSV file not found at path: {csv_file_path}")
        return None
    except Exception as e:
        # print(f"An error occurred: {e}")
        return None

def get_dst_start_end(year, latitude, longitude):
    # Get the timezone for the given latitude and longitude
    tf = TimezoneFinder()
    timezone_str = tf.timezone_at(lat=latitude, lng=longitude)
    
    if timezone_str is None:
        raise ValueError("Could not find timezone for the given coordinates.")
    
    # Get the timezone object
    timezone = pytz.timezone(timezone_str)
    
    # Define the dates for the beginning and end of the year (naive datetime)
    start_of_year = datetime(year, 1, 1)
    end_of_year = datetime(year, 12, 31)
    
    dst_start = None
    dst_end = None

    # Start by localizing the first date
    previous_offset = timezone.localize(start_of_year).dst()

    # Loop through each day of the year
    for dt in [start_of_year + timedelta(days=i) for i in range((end_of_year - start_of_year).days + 1)]:
        localized_dt = timezone.localize(dt)  # Localize naive datetime
        current_offset = localized_dt.dst()
        
        if previous_offset == timedelta(0) and current_offset != timedelta(0):
            dst_start = localized_dt
        elif previous_offset != timedelta(0) and current_offset == timedelta(0):
            dst_end = localized_dt
            break
        
        previous_offset = current_offset
    
    return dst_start, dst_end

# Function to calculate the distance between two points given their latitudes and longitudes
def haversine_distance(lat1, lon1, lat2, lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1 
    dlon = lon2 - lon1 
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    r = 6371  # Radius of Earth in kilometers. Use 3956 for miles. Determines return value units.

    #Distance returned in km
    return c * r

def find_closest_design_condition(lat,lon,design_conditions_file):
    """
    Finds the closest design condition from the CSV file based on the given latitude and longitude.

    Parameters:
    lat (float): The latitude of the target location.
    lon (float): The longitude of the target location.
    csv_file (str): The path to the CSV file containing design conditions.

    Returns:
    str: The 2021 design condition string for the closest location.
    """
    
    # Read the CSV file
    df = pd.read_csv(design_conditions_file)

    # Calculate distance from target coordinates to each row in the dataframe
    df['distance'] = df.apply(lambda row: haversine_distance(lat, lon, row['latitude'], row['longitude']), axis=1)

    # Find the row with the minimum distance
    closest_row = df.loc[df['distance'].idxmin()]

    # Return the design conditions for 2021
    return closest_row['2021_design_conditions']

def retrieve_distance_station_location(wmo, lat_location, lon_location):
    meteostat_stations = pd.read_csv('resources/meteostat_stats.csv', index_col='id')
    lat_station = meteostat_stations[meteostat_stations.index == wmo]['latitude'].values[0]
    lon_station = meteostat_stations[meteostat_stations.index == wmo]['longitude'].values[0]
    distance_km = haversine_distance(lat_location, lon_location, lat_station, lon_station)
    distance_mi = distance_km*0.621371
    return distance_mi




# Define constants
year = 2022
file_type = 'AMY'
save_folder = f'epws_wmo_{year}_test'

data_file = f'resources/zip_code_list_{year}_test.csv'
zipcodes = pd.read_csv(data_file, dtype={f'EPW_file_name_{year}': str, f'weather_station_wmo_{year}': str})

stations_df = pd.read_csv('resources/meteostat_stats.csv', index_col='id')

for index, row in zipcodes.iterrows():
    lat, lon = row['lat'], row['lng']
    save_name = None
    retrieve_status, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info = run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    
    if retrieve_info:
        retrieve_status, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)
    
    if not isinstance(retrieve_status, bool):
        raise TypeError(f"retrieve_status is not a boolean: {retrieve_status}")
    
    distance_mi = retrieve_distance_station_location(wmo, lat, lon, stations_df)
    update_if_missing(zipcodes, index, f"EPW_file_name_{year}", retrieve_status)
    update_if_missing(zipcodes, index, f"distance_location_station_miles_{year}", distance_mi)
    update_if_missing(zipcodes, index, f"weather_station_wmo_{year}", wmo)
    update_if_missing(zipcodes, index, f"hdd_base65F_{year}", hdd)
    update_if_missing(zipcodes, index, f"cdd_base65F_{year}", cdd)

zipcodes.to_csv(data_file, index=False)


0
99638
1
99923


2
99776


3
99566


4
99780
5
89017
6
59058
7
57651


8
59324
9
86510
10
57720


11
89043
12
89832
13
99667


14
57634
15
57724
16
86039
17
85341


KeyboardInterrupt: 

In [ ]:
# Let's first create the dataframe with entries we have clearly from images provided by the user:
import pandas as pd

# Papers extracted from provided images
data = [
    {
        "Paper (#)": 3,
        "Title": "Control of electric thermal storage under real time pricing",
        "Authors": "B. Daryanian, Roger E. Bohn, Richard D. Tabors",
        "Insights": "Electric rate structures, particularly real-time pricing (RTP), significantly impact thermal energy storage (ETS) systems allowing charging during lowest cost periods...",
        "Methods Used": "Implementation of real-time pricing (RTP), comparison with time-of-use (TOU) control through experiment",
        "Results": "Nearly 50% increased benefits with RTP compared to TOU control",
        "Findings": "RTP significantly increases ETS systems' benefits to utilities compared to TOU",
        "Objectives": "Evaluate effectiveness of RTP as a demand-side management for ETS"
    },
    {
        "Paper (#)": 4,
        "Title": "Sizing of electric thermal storage under real time pricing",
        "Authors": "B. Daryanian, Roger E. Bohn",
        "Insights": "RTP significantly impacts TES sizing and operation for optimized electricity consumption",
        "Methods Used": "Simulation study analyzing storage sizes and optimization algorithms under RTP",
        "Results": "Larger storage sizes under RTP lead to reduced utility service costs",
        "Findings": "Larger storage sizes beneficial but show diminishing returns at high capacities under RTP",
        "Objectives": "Analyze RTP impact on TES sizing and utility cost savings"
    },
    {
        "Paper (#)": 5,
        "Title": "Commercial feasibility of thermal storage in buildings for utility load leveling",
        "Authors": "J.G. Asbury, R.F. Giese, R.O. Mueller",
        "Insights": "Electric rates and incentives are key for commercial viability of thermal storage",
        "Methods Used": "Economic viability analysis across four different utility service areas",
        "Results": "Thermal storage systems cost-effective particularly aligned with suitable rate structures",
        "Findings": "Broad applicability and effectiveness linked to season and regional demand patterns",
        "Objectives": "Evaluate cost-effectiveness for utility load leveling using thermal storage systems"
    },
    {
        "Paper (#)": 6,
        "Title": "Leveraging thermal storage to cut the electricity bill for datacenter cooling",
        "Authors": "Yefu Wang, Xiaorui Wang, Yanwei Zhang",
        "Insights": "Utilization of TStore and thermal masses reduces cooling costs significantly",
        "Methods Used": "CFD simulation, overcooling, auxiliary thermal storage tanks",
        "Results": "16.8% electricity bill reduction achieved compared to traditional methods",
        "Findings": "Effective reduction in bills with maintained cooling performance",
        "Objectives": "Design and evaluate TStore leveraging variable electricity prices for optimized cooling"
    },
    {
        "Paper (#)": 9,
        "Title": "A Comprehensive Building Load Optimization Method from Utility Rate Structure Perspective with Renewables and Energy Storage",
        "Authors": "A S M Jahid Hasan, Luis Fernando Enriquez-Contreras, Jubair Yusuf",
        "Insights": "Addresses optimization of loads and BESS integration considering rate structures",
        "Methods Used": "Comprehensive optimization model addressing diversity in utility rate structures",
        "Results": "BESS beneficial for buildings under CPP structures",
        "Findings": "BESS systems particularly advantageous for CPP structures and low load factor buildings",
        "Objectives": "Develop optimization model for informed decision making for renewables and storage"
    },
    {
        "Paper (#)": 10,
        "Title": "Using the thermal energy storage potential of residential homes for ToU rate savings and demand response",
        "Authors": "Steven Wong, Nicolas Levesque, Véronique Delisle",
        "Insights": "ToU rates influence residential TES effectiveness",
        "Methods Used": "Four-capacitance model simulations using Matlab, various thermostat strategies evaluated",
        "Results": "Moderate cost savings achievable through TES with advanced thermostat strategies",
        "Findings": "TES and smart thermostats achieve cost savings under ToU rates with minimal discomfort",
        "Objectives": "Simulate DR strategies for residential TES, assess comfort and consumption impacts"
    },
    {
        "Paper (#)": 13,
        "Title": "Design and performance of a long duration electric thermal energy storage demonstration plant at megawatt-scale",
        "Authors": "Jan Eggers, Michael von der Heyde, Sören Hendrik Thaele",
        "Insights": "",
        "Methods Used": "Electric thermal energy storage (ETES), horizontal-flow design, low-cost natural rocks",
        "Results": "",
        "Findings": "",
        "Objectives": ""
    },
    {
        "Paper (#)": 14,
        "Title": "Cost-effective Electro-Thermal Energy Storage to balance small scale renewable energy systems",
        "Authors": "Sampson Tetteh, Maryam Roza Yazdani, Annukka Santasalo-Aarnio",
        "Insights": "",
        "Methods Used": "Modular electro-thermal ETES with thermal oil, molten salt, sand, Stirling engines, electric heater",
        "Results": "",
        "Findings": "",
        "Objectives": ""
    },
    {
        "Paper (#)": 18,
        "Title": "ThermalThrift: Cost Effective Thermal Energy Storage for Load Shifting with Water Heaters",
        "Authors": "Erin Griffiths, Kamin Whitehouse",
        "Insights": "",
        "Methods Used": "ThermalThrift system, water usage data collection, TOU pricing schedules analysis",
        "Results": "",
        "Findings": "",
        "Objectives": ""
    },
    {
        "Paper (#)": 19,
        "Title": "Electrically charged thermal energy storage systems for grid-level electricity storage",
        "Authors": "Laureen Meroueh",
        "Insights": "",
        "Methods Used": "Evaluation of chemical, thermal, mechanical storage, focus on PCMs, two system designs proposed",
        "Results": "",
        "Findings": "",
        "Objectives": ""
    },
    {
        "Paper (#)": 20,
        "Title": "Thermo-economic design of an electric heater to store renewable curtailment in solar power tower plants",
        "Authors": "D. Pardillos-Pobo, P.A. González-Gómez, M. Laporte-Azcué",
        "Insights": "",
        "Methods Used": "Electric heater in solar power tower plant design optimized for LCOS",
        "Results": "",
        "Findings": "",
        "Objectives": ""
    },
    {
        "Paper (#)": 11,
        "Title": "Thermal Energy Storage for Electricity Peak-demand Mitigation: A Solution in Developing and Developed World Alike",
        "Authors": "Nicholas DeForest, Goncalo Mendes, Michael Stadler",
        "Insights": "Simulation with DER-CAM model to optimize TES sizing for office buildings in diverse climates.",
        "Methods Used": "DER-CAM simulation across Miami, Lisbon, Shanghai, Mumbai; EnergyPlus 7.0 for building simulations",
        "Results": "",
        "Findings": "",
        "Objectives": "Evaluate TES effectiveness in reducing peak electricity demand globally."
    },
    {
        "Paper (#)": 12,
        "Title": "The Price is Right? Encouraging Load Shifting with Time of Use Rates",
        "Authors": "Ellen Franconi, Xuechen Lei, Wooyoung Jung",
        "Insights": "Optimization of battery system discharge using extensive TOU rate analysis.",
        "Methods Used": "Building simulation analysis, OpenEI rate database",
        "Results": "",
        "Findings": "Critical role of demand charges in battery storage economics",
        "Objectives": "Assess economics and incentive levels required for battery storage viability with TOU rates."
    },
    {
        "Paper (#)": 15,
        "Title": "Reducing energy costs and minimizing capital requirements: case studies of thermal energy storage (tes)",
        "Authors": "John S. Andrepont",
        "Insights": "",
        "Methods Used": "",
        "Results": "",
        "Findings": "",
        "Objectives": ""
    },
    {
        "Paper (#)": 16,
        "Title": "Addressing energy storage needs at lower cost via on-site thermal energy storage in buildings",
        "Authors": "Adewale Odukomaiya, Jason Woods, Nelson James",
        "Insights": "Framework to calculate LCOS comparing TES with lithium-ion batteries; includes load projections for 2050",
        "Methods Used": "LCOS calculation, load projections, phase change materials (PCM) configurations",
        "Results": "",
        "Findings": "",
        "Objectives": "Enhance TES efficiency and effectiveness with HVAC-integrated PCM"
    },
    {
        "Paper (#)": 17,
        "Title": "Cost optimal operation of thermal energy storage system with real-time prices",
        "Authors": "Toru Kashima, Stephen Boyd",
        "Insights": "Optimization of TES operation using MILP considering future demands and electricity pricing",
        "Methods Used": "Mixed integer linear programming (MILP), branch and bound algorithm",
        "Results": "",
        "Findings": "Efficient optimization achievable through MILP formulation",
        "Objectives": "Optimize TES system operation economically under real-time pricing"
    },
    {
        "Paper (#)": 3,
        "Title": "Control of electric thermal storage under real time pricing",
        "Authors": "B. Daryanian, Roger E. Bohn, Richard D. Tabors",
        "Insights": "Electric rate structures, particularly real-time pricing (RTP), significantly impact thermal energy storage (ETS) systems allowing charging during lowest cost periods...",
        "Methods Used": "Implementation of real-time pricing (RTP), comparison with time-of-use (TOU) control through experiment",
        "Results": "Nearly 50% increased benefits with RTP compared to TOU control",
        "Findings": "RTP significantly increases ETS systems' benefits to utilities compared to TOU",
        "Objectives": "Evaluate effectiveness of RTP as a demand-side management for ETS"
    },
    {
        "Paper (#)": 4,
        "Title": "Sizing of electric thermal storage under real time pricing",
        "Authors": "B. Daryanian, Roger E. Bohn",
        "Insights": "RTP significantly impacts TES sizing and operation for optimized electricity consumption",
        "Methods Used": "Simulation study analyzing storage sizes and optimization algorithms under RTP",
        "Results": "Larger storage sizes under RTP lead to reduced utility service costs",
        "Findings": "Larger storage sizes beneficial but show diminishing returns at high capacities under RTP",
        "Objectives": "Analyze RTP impact on TES sizing and utility cost savings"
    },
    {
        "Paper (#)": 1,
        "Title": "Electric rate structures for thermal energy storage evaluation",
        "Authors": "D.R. Brown, S.M. Garrett, J.M. Sedgewick",
        "Insights": "Electric rate structures significantly impact TES by influencing design and economic viability, particularly RTP.",
        "Methods Used": "General approach involving industry consultation, statistical analysis",
        "Results": "Characterized range of utility rate structures for TES",
        "Findings": "Different rate structures affect economic viability of TES; multiple rate scenarios necessary for accuracy",
        "Objectives": "Identify optimal utility rate structures for economic feasibility of TES"
    },
    {
        "Paper (#)": 2,
        "Title": "Control of Thermal Energy Storage in Commercial Buildings for California Utility Tariffs and Demand Response",
        "Authors": "Rongxin Yin, Doug Black, Piette Mary A.",
        "Insights": "Electric rates significantly impact TES system economics, particularly regarding demand charges",
        "Methods Used": "EnergyPlus simulations, analysis under DR programs",
        "Results": "Partial storage TES systems more effective for DR, offering quicker payback periods",
        "Findings": "Partial storage systems favorable for moderate rate incentives and smaller customers",
        "Objectives": "Develop simulations for assessing TES integration into DR programs under California tariffs"
    },
    {
        "Paper (#)": 7,
        "Title": "Can storage reduce electricity consumption? A general equation for the grid-wide efficiency impact of using cooling thermal energy storage for load shifting",
        "Authors": "Thomas A. Deetjen, Andrew Reimers, Michael E. Webber",
        "Insights": "Focuses on the efficiency gains from using Cooling Thermal Energy Storage (CTES) for load shifting to reduce grid-wide energy consumption.",
        "Methods Used": "Efficiency modeling, graphical sensitivity analysis",
        "Results": "Significant potential for energy savings and reduction in primary fuel consumption demonstrated",
        "Findings": "CTES can enhance grid-wide efficiency under optimal conditions",
        "Objectives": "Estimate grid-wide energy savings potential of CTES"
    },
    {
        "Paper (#)": 8,
        "Title": "Implications of Rate Design for the Customer-Economics of Behind-the-Meter Storage",
        "Authors": "Naim Darghouth, Galen Barbose, Andrew Mills",
        "Insights": "Rate designs such as TOU, CPP, and RTP significantly influence economics of behind-the-meter storage systems.",
        "Methods Used": "Perfect foresight dispatch algorithm using HOMER, evaluation of rate designs",
        "Results": "Demand charge savings vary significantly based on rate design and storage duration",
        "Findings": "Longer duration storage yields greater arbitrage savings compared to demand charge savings",
        "Objectives": "Investigate the impact of rate designs on economics and bill savings of behind-the-meter storage"
    },
    {
        "Paper (#)": 16,
        "Title": "Electric tariffs and thermal energy storage systems for buildings",
        "Authors": "Ahmet Fertelli",
        "Insights": "",
        "Methods Used": "Examined ten-year price changes of heating fuels, analyzed hourly electricity consumption during heating season",
        "Results": "Thermal energy storage systems 20-40% less costly until 2020; 40-55% less costly than natural gas in 2021-2022",
        "Findings": "Thermal energy storage reduces costs by 20-55%",
        "Objectives": ""
    },
    {
        "Paper (#)": 17,
        "Title": "Optimum storage size for thermal energy storage system",
        "Authors": "A.H. Kassim, Mohamad Taib Miskon, Ilham Rustam",
        "Insights": "",
        "Methods Used": "Cost evaluation using C1 and C2 tariff comparison; assessment of extra load, energy, cost from TES",
        "Results": "TES best with 100% storage capacity; C2 tariff provides optimum cost saving",
        "Findings": "Improved TES performance and cost-effectiveness with optimal tariffs",
        "Objectives": ""
    },
    {
        "Paper (#)": 18,
        "Title": "Thermal Energy Storage Systems",
        "Authors": "Ehsan Mohseni Languri, Glenn Cunningham",
        "Insights": "",
        "Methods Used": "Sensible, latent, and thermo-chemical TES systems; reversible chemical reactions",
        "Results": "Overview of TES types; challenges and potential applications discussed",
        "Findings": "Improved thermal conductivity enhances response; form-stable composites minimize PCM leakage",
        "Objectives": ""
    },
    {
        "Paper (#)": 19,
        "Title": "Thermal Energy Storage: Solutions for Demand Management",
        "Authors": "John S. Andrepont",
        "Insights": "",
        "Methods Used": "Review of TES technologies, case histories",
        "Results": "TES provides flexibility for cooling; economic benefits realized",
        "Findings": "TES offers flexibility and economic benefits; distinct advantages and limitations of various technologies",
        "Objectives": ""
    },
    {
        "Paper (#)": 20,
        "Title": "The Influence of Different Network Tariffs on Distribution Grid Reinforcement Costs",
        "Authors": "L. Kundert, A. Heider, Gabriela Hug",
        "Insights": "",
        "Methods Used": "Cost-minimizing consumer-based optimization, case study of six grids in Germany",
        "Results": "Time-varying tariffs can increase grid reinforcement costs; constant tariffs reduce costs by 9.5%",
        "Findings": "Time-varying tariffs increase costs; constant tariffs reduce costs by 9.5%",
        "Objectives": ""
    },
    {
        "Paper (#)": 11,
        "Title": "Thermal Energy Storage (TES): Optimizing the Economics of Energy and Capital",
        "Authors": "John S. Andrepont",
        "Insights": "",
        "Methods Used": "Analysis of large capacity TES applications and trends; examination of energy efficiency and economic benefits",
        "Results": "Significant energy efficiency gains; large capital cost savings compared to conventional systems",
        "Findings": "Large capacity TES installations show significant energy efficiency gains; lower supply temperatures optimize system performance and reduce costs",
        "Objectives": ""
    },
    {
        "Paper (#)": 12,
        "Title": "Potenziale der Integration thermischer Energiespeicher in Dampfkraftwerke",
        "Authors": "Thomas Loeper, Michael Krüger, Marcel Richter",
        "Insights": "",
        "Methods Used": "Development and evaluation of storage integration concepts; stationary system simulations for net power output changes",
        "Results": "Minimum load reduction up to 4% during charging; load increase up to 5% during discharging",
        "Findings": "Three leading concepts for thermal energy storage integration identified; efficiency improvements through simulations",
        "Objectives": ""
    },
    {
        "Paper (#)": 13,
        "Title": "Impact of tariff structures on energy community and grid operational parameters",
        "Authors": "Bodan Velkovski, Vladimir Gjorgievski, Despoina Kothona",
        "Insights": "",
        "Methods Used": "Mixed-integer linear programming optimization; analysis of different tariff structures for energy sharing incentives",
        "Results": "Energy sharing reduces grid losses, peak loads, reverse flows; dynamic tariffs yield favorable results",
        "Findings": "Dynamic tariffs and capacity payments enhance community energy sharing benefits",
        "Objectives": ""
    },
    {
        "Paper (#)": 14,
        "Title": "Control of Thermal Energy Storage in Commercial Buildings for California Utility Tariffs and Demand Response",
        "Authors": "Rongxin Yin, Doug Black, Mary A. Piette",
        "Insights": "",
        "Methods Used": "Analytical model EnergyPlus for TES and DR programs; case studies on utility DR program changes and TES usage",
        "Results": "Evaluated TES effectiveness and identified optimal TES configurations for different buildings and climates",
        "Findings": "Partial storage systems better for demand response participation",
        "Objectives": ""
    },
    {
        "Paper (#)": 15,
        "Title": "Impact of advanced electricity tariff structures on the optimal design, operation, and profitability of a grid-connected PV system with energy storage",
        "Authors": "Lionel Bloch, Jordan Holweger, Christophe Ballif",
        "Insights": "",
        "Methods Used": "Mixed-integer linear programming for PV and battery systems; benchmarking five tariff scenarios",
        "Results": "Block rate tariff most promising for optimal design; capacity-based tariffs rely on PV curtailment for generation peaks",
        "Findings": "Block rate tariff optimal; capacity-based tariffs beneficial for managing generation peaks",
        "Objectives": ""
    },
    {
        "Paper (#)": 1,
        "Title": "Optimal Storage Response to Utility Tariff Structures and Potential Use of Capacity Charges",
        "Authors": "Killian McKenna",
        "Insights": "Focuses primarily on lithium-ion systems, not specifically TES.",
        "Methods Used": "Linear optimization for storage tariff response; customer net load metrics analysis",
        "Results": "Optimal storage response examined; capacity charges reduce peak imports and exports",
        "Findings": "Optimal storage response varies with tariff structures; capacity charges beneficial",
        "Objectives": "Analyze optimal storage response and net load metrics under diverse tariffs"
    },
    {
        "Paper (#)": 2,
        "Title": "Thermal Energy Storage for Electricity Peak-demand Mitigation: A Solution in Developing and Developed World Alike",
        "Authors": "Nicholas DeForest, Goncalo Mendes, Michael Stadler",
        "Insights": "TES deployment influenced significantly by tariff structures.",
        "Methods Used": "Simulation with DER-CAM model across four cities",
        "Results": "TES reduces peak electricity consumption by 30-38%; substantial savings",
        "Findings": "TES effectively reduces peak electricity demand",
        "Objectives": "Simulate TES effectiveness for peak electricity demand mitigation"
    },
    {
        "Paper (#)": 3,
        "Title": "A comprehensive review on mobilized thermal energy storage",
        "Authors": "Shanmuga Sundaram Anandan, Jagannathan Sunderababu",
        "Insights": "Primarily addresses industrial waste heat recovery.",
        "Methods Used": "Review on industrial waste heat recovery and mobilized thermal energy storage",
        "Results": "Industrial applications waste significant heat resources; waste heat recovery solutions effective",
        "Findings": "Industrial waste heat recovery significantly beneficial",
        "Objectives": "Review industrial waste heat recovery and mobilized thermal storage solutions"
    },
    {
        "Paper (#)": 4,
        "Title": "Integration of curtailed wind into flexible electrified heating networks with demand-side response and thermal storage",
        "Authors": "Andrew Francis Lyden, Daniel Friedrich",
        "Insights": "Tariff structures significantly influence TES economic viability integrating curtailed wind.",
        "Methods Used": "Integration of curtailed wind, mathematical optimization of electrified heat plant",
        "Results": "Large-scale storage lowers costs and increases curtailed wind integration",
        "Findings": "Large-scale storage reduces costs; financial incentives needed",
        "Objectives": "Analyze integration of curtailed wind into heating networks"
    },
    {
        "Paper (#)": 5,
        "Title": "The potential of heat storage as variation management in the electricity and district heating sectors",
        "Authors": "Petra Holmer, Jonathan Ullmark",
        "Insights": "Examines thermal energy storage in electricity and district heating sectors.",
        "Methods Used": "Greenfield cost-optimizing model; scenario analysis for district heating and thermal energy storage",
        "Results": "Impacts cost-optimal DH system; promotes wind power, reduces curtailment",
        "Findings": "Significant impact on DH system composition; cost and curtailment reduction",
        "Objectives": "Explore thermal storage as variation management; analyze optimal district heating integration"
    },
    {
        "Paper (#)": 6,
        "Title": "Optimizing the Capacity of Thermal Energy Storage in Industrial Clusters",
        "Authors": "Mandar Thombre, Sandeep Prakash, Brage Rugstad Knudsen",
        "Insights": "Focuses on optimizing TES capacity using scenario-based stochastic programming for industrial clusters.",
        "Methods Used": "Single-level and bilevel formulation minimizing design and operation costs",
        "Results": "Two formulations for optimal TES capacity presented and compared via industrial case study",
        "Findings": "Optimal TES design improves energy efficiency in clusters",
        "Objectives": "Optimize TES design and minimize combined design-operation cost"
    },
    {
        "Paper (#)": 7,
        "Title": "Current, Projected Performance and Costs of Thermal Energy Storage",
        "Authors": "Laura Pompe, Fabio Nardecchia, Adio Milozzi",
        "Insights": "Examines performance, costs, and applications of TES technologies.",
        "Methods Used": "Overview of sensible, latent, and thermochemical TES applications",
        "Results": "TES market projected at 10.1 billion USD by 2027",
        "Findings": "Sensible heat TES dominates due to affordability and wide applications",
        "Objectives": "Review current TES performance, costs, and analyze European applications"
    },
    {
        "Paper (#)": 8,
        "Title": "Integration of curtailed wind into flexible electrified heating networks with demand-side response and thermal storage",
        "Authors": "Andrew Francis Lyden, Daniel Friedrich",
        "Insights": "Highlights importance of tariff structures for integrating curtailed wind via TES.",
        "Methods Used": "Mathematical optimization of electrified heat plant designs",
        "Results": "Large-scale TES lowers integration costs, financial incentives essential",
        "Findings": "Thermal storage increases curtailed wind integration",
        "Objectives": "Investigate wind integration into heating networks, analyze curtailment mitigation"
    },
    {
        "Paper (#)": 9,
        "Title": "Untapping Industrial Flexibility via Waste Heat-Driven Pumped Thermal Energy Storage Systems",
        "Authors": "Stefano Barberis, Simone Maccarini, S. Shamsi",
        "Insights": "Explores industrial waste heat utilization via pumped thermal energy storage (PTES).",
        "Methods Used": "Feasibility study of sCO2-based PTES, industrial heat valorization",
        "Results": "Enhanced PTES performance, CAPEX reduced with sCO2 integration",
        "Findings": "Waste heat significantly improves PTES efficiency",
        "Objectives": "Enhance PTES efficiency, improve industrial energy competitiveness"
    },
    {
        "Paper (#)": 10,
        "Title": "Developments in Thermal Energy Storage: Large Applications, Low Temps, High Efficiency, and Capital Savings",
        "Authors": "John S. Andrepont",
        "Insights": "Discusses efficiency, large-scale applications, and economics of TES.",
        "Methods Used": "Economic studies, analysis of large-scale TES systems",
        "Results": "Increased application and documented economic benefits of large TES",
        "Findings": "Large-scale TES improves efficiency and economic viability",
        "Objectives": "Discuss TES trends, explore efficiency and economic benefits"
    },
    {
        "Paper (#)": 6,
        "Title": "Performance evaluation of advanced energy storage systems: a review",
        "Authors": "Gulam Sméani, Muhammad Reman Islam, Ahmad Naim Ahmad Yahaya",
        "Insights": "PHES and CAES are the most cost-effective energy storage systems evaluated in terms of $/kWh.",
        "Methods Used": "Evaluation of energy storage systems on various parameters; comparison of different storage technologies",
        "Results": "Supercapacitors highest power density; Hydrogen fuel cells highest energy density",
        "Findings": "Performance evaluated for various storage systems; significant differences highlighted",
        "Objectives": "Evaluate performance and economic viability of advanced energy storage systems"
    },
    {
        "Paper (#)": 7,
        "Title": "The Impact of Cost and Energy Storage on Power Sector Decarbonisation",
        "Authors": "Subhadip Bhattacharya, Rangan Banerjee, Venkatasallanathan Ramadesigan",
        "Insights": "Declining solar and battery storage costs significantly impact decarbonization efforts.",
        "Methods Used": "Bottom-up TIMES-based cost optimization; incorporating power generation, energy storage, green hydrogen",
        "Results": "Energy storage projected to supply 20% electricity by 2070",
        "Findings": "Non-fossil power sector achievable; emissions peak required by 2035",
        "Objectives": "Evaluate declining costs impact and future net-zero portfolio"
    },
    {
        "Paper (#)": 8,
        "Title": "Economic top-down evaluation of the costs of energy storages—A simple economic truth in two equations",
        "Authors": "Christoph Rathgeber, Eberhard Livemann, Andreas Hauer",
        "Insights": "Economic viability of energy storage dependent on cycle frequency and costs.",
        "Methods Used": "Economic top-down approach; present value annuity factor calculation",
        "Results": "Higher cycle frequency allows higher acceptable costs",
        "Findings": "Acceptable costs depend significantly on user's economic conditions",
        "Objectives": "Analyze economic viability and cycle frequency impacts"
    },
    {
        "Paper (#)": 9,
        "Title": "Rapid visualization of the potential residential cost savings from energy storage under time-of-use electric rates",
        "Authors": "Michael Lanahan, Sarah Engert, Taewoo Kim",
        "Insights": "Energy storage significantly reduces residential peak load costs.",
        "Methods Used": "EnergyPlus load generation; Matlab post-processing; Google Fusion Tables",
        "Results": "$420 potential annual savings per home with recommended 24 kWh capacity",
        "Findings": "Energy storage enhances flexibility and reduces peak demand",
        "Objectives": "Visualize cost savings and analyze TOU rate impacts"
    },
    {
        "Paper (#)": 10,
        "Title": "An economic optimization method and system for energy storage cost and benefit",
        "Authors": "Xue Jinhua, Yang Bo, Shi Ruxin",
        "Insights": "Economic optimization for maximizing net income of energy storage lifecycle in photovoltaics.",
        "Methods Used": "Particle swarm optimization; optimization objective and constraint construction",
        "Results": "Maximizes total net income; analyzes income-expenditure in photovoltaics",
        "Findings": "Algorithm effectively maximizes storage net income",
        "Objectives": "Optimize economic performance of storage in photovoltaic applications"
    },
    {
        "Paper (#)": 1,
        "Title": "kWh Cost Analysis of Energy Storage Power Station Based on Changing Trend of Battery Cost",
        "Authors": "Hu Jianhua, Peng Zhe, Zhenyu Zhao",
        "Insights": "Current electrochemical energy storage costs range from 0.6 to 0.9 yuan/(kW·h).",
        "Methods Used": "Cost calculation of energy storage per kWh; technical analysis of battery cost trends",
        "Results": "Current electrochemical energy storage cost: 0.6 to 0.9 yuan/kWh; Target cost for widespread application: 0.3 to 0.4 yuan/kWh",
        "Findings": "Current electrochemical energy storage cost: 0.6 to 0.9 yuan/kWh; Target cost for widespread application: 0.3 to 0.4 yuan/kWh",
        "Objectives": "Calculate cost per kilowatt-hour of energy storage; analyze battery cost trends and their impact"
    },
    {
        "Paper (#)": 2,
        "Title": "Energy storage as part of electricity distribution asset management",
        "Authors": "Juha Haakana, Jouni Haapaniemi, Jukka Lassila",
        "Insights": "",
        "Methods Used": "Feasibility analysis of energy storages in Finnish rural area; cost analysis to enable wide use of energy storages",
        "Results": "ES can avoid expensive underground cable investments; ES unit costs should be below 200 €/kWh for wide use",
        "Findings": "Energy storage can prevent long interruptions in substations; unit costs should be below 200 €/kWh for feasibility",
        "Objectives": ""
    },
    {
        "Paper (#)": 3,
        "Title": "Energy storage as part of electricity distribution asset management",
        "Authors": "Juha Haakana, Jouni Haapaniemi, Jukka Lassila",
        "Insights": "",
        "Methods Used": "Feasibility analysis of energy storages in Finnish rural area; cost analysis to enable wide use of energy storages",
        "Results": "ES as back-up avoids interruptions, reduces expensive cable investments; storage unit costs should be below 200 €/kWh for wide use",
        "Findings": "Energy storage can prevent long interruptions in medium-voltage networks; unit costs should be below 200 €/kWh for wide use",
        "Objectives": ""
    },
    {
        "Paper (#)": 4,
        "Title": "The Cost of Renewable Electricity and Energy Storage in Germany",
        "Authors": "Nico Wehrle",
        "Insights": "Electricity generation costs in Germany range from 0.02 to 0.10 EUR/kWh.",
        "Methods Used": "Calculation model with real data of German electricity generation; Levelised Cost of Storage (LCOS) metric for energy storage costs",
        "Results": "Total system costs range from 0.19 to 0.28 EUR/kWh; energy storage is more significant than electricity generation in terms of costs",
        "Findings": "Storage costs exceed electricity generation costs in renewable systems; total system costs range from 0.19 to 0.28 EUR/kWh",
        "Objectives": ""
    },
    {
        "Paper (#)": 5,
        "Title": "Energy Storage Control Algorithms to Reduce the Cost of Electric Energy to Consumers",
        "Authors": "Alexander I. Orlov, V. T. Sidorova, Kirill A. Samoilov",
        "Insights": "The paper discusses energy storage systems, particularly lithium-ion batteries, which can reduce electricity costs by implementing control algorithms.",
        "Methods Used": "Reducing peak power through control algorithms; minimizing mains electricity consumption during control hours",
        "Results": "First algorithm reduces peak power costs effectively; second algorithm minimizes power charge to zero",
        "Findings": "Algorithms reduce electricity costs for specific price categories; energy storage capacity influences cost reduction effectiveness",
        "Objectives": ""
    }
]

# Creating DataFrame and saving as CSV
df_papers_complete = pd.DataFrame(data)

# Save as CSV file
csv_path_complete = 'TES_ratestructures.csv'
df_papers_complete.to_csv(csv_path_complete, index=False)

csv_path_complete


'TES_ratestructures.csv'

In [2]:
# Let's create the table again using all the provided screenshots

data = [
    {
        "Paper": "Commercial feasibility of thermal storage in buildings for utility load leveling",
        "Authors": "J.G. Asbury, R.F. Giese, R.O. Mueller",
        "Insights": "Implementation of electric rates and incentives crucial for TES commercialization in buildings.",
        "Methods Used": "Evaluates cost-effectiveness for utility load leveling across various service areas.",
        "Results": "Cost-effective for space heating in winter; air conditioning cost-effectiveness varies by cooling season length.",
        "Findings": "Storage systems' commercialization depends on favorable electric rates and incentives.",
        "Objectives": "Evaluate cost-effectiveness specifically for utility load leveling."
    },
    {
        "Paper": "Leveraging thermal storage to cut the electricity bill for datacenter cooling",
        "Authors": "Yefu Wang, Xiaorui Wang, Yanwei Zhang",
        "Insights": "TES systems like TStore can effectively cut data center cooling costs.",
        "Methods Used": "CFD simulations for cooling strategies in data centers.",
        "Results": "16.8% reduction in cooling electricity bill.",
        "Findings": "TES effectively reduces electricity costs without causing overheating.",
        "Objectives": "Develop TStore to reduce datacenter cooling costs."
    },
    {
        "Paper": "Reducing energy costs and minimizing capital requirements: case studies of TES",
        "Authors": "John S. Andrepont",
        "Insights": "-",
        "Methods Used": "-",
        "Results": "-",
        "Findings": "-",
        "Objectives": "-"
    },
    {
        "Paper": "Addressing energy storage needs at lower cost via on-site TES in buildings",
        "Authors": "Adewale Odukomaiya, Jason Woods, Nelson James",
        "Insights": "Framework for LCOS calculation for TES with electrical storage comparisons.",
        "Methods Used": "Projections of thermal and non-thermal loads for buildings in 2050.",
        "Results": "Considers configurations and PCMs for HVAC-integrated TES systems.",
        "Findings": "-",
        "Objectives": "-"
    },
    {
        "Paper": "Thermal Energy Storage for Electricity Peak-demand Mitigation",
        "Authors": "Nicholas DeForest, Goncalo Mendes, Michael Stadler",
        "Insights": "-",
        "Methods Used": "DER-CAM and EnergyPlus simulations for building performance across global contexts.",
        "Results": "-",
        "Findings": "-",
        "Objectives": "-"
    },
    {
        "Paper": "The Price is Right? Encouraging Load Shifting with Time of Use Rates",
        "Authors": "Ellen Franconi, Xuechen Lei, Wooyoung Jung",
        "Insights": "-",
        "Methods Used": "Building simulations to optimize battery discharge under TOU rates.",
        "Results": "-",
        "Findings": "-",
        "Objectives": "-"
    }
]

# Convert the list of dictionaries to a DataFrame
df_new_entries = pd.DataFrame(data)

# Save the DataFrame to a CSV file
csv_path = 'TES_ratestructures.csv'
df_new_entries.to_csv(csv_path, index=False)

csv_path


'TES_ratestructures.csv'